# Clase 08 — Tests de hipótesis II: comparar varios grupos

**Curso:** Estadística Aplicada con Python · **Bloque 3:** Inferencia
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/08_tests_hipotesis_ii.md`

## Qué vas a ver en esta clase

- **El problema de los t-tests a pares** cuando hay más de dos grupos.
- **ANOVA de un factor** (`f_oneway`): la idea (varianza entre y dentro de los grupos), la tabla ANOVA a mano y el tamaño del efecto **η²**.
- **Comparaciones posteriores (post-hoc) de Tukey**: un ANOVA significativo no dice *qué* grupos difieren.
- **Supuestos**: normalidad de los residuos (gráfico Q-Q, Shapiro-Wilk), homogeneidad de varianzas (Levene) y qué hacer si fallan (Welch-ANOVA).
- **Tests no paramétricos**: Mann-Whitney y Kruskal-Wallis, y cuándo ganan al ANOVA (simulación de potencia con datos de cola larga).
- **Leer y comunicar** un resultado completo (F, p, η², post-hoc) y una tabla para **elegir el test**.

## Requisitos

- Clase 7 (lógica del test, p-value, errores de tipo I y II, t-test de Welch, tamaño del efecto) y clase 6 (error estándar, intervalos de confianza).
- Librerías: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra. Todos los datos (tiempos de preparación de pedidos, tiempos de espera) son sintéticos: un *ejemplo ilustrativo*, no datos reales. Las categorías (chi-cuadrado) y las comparaciones múltiples se ven en la clase 9.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla**: con la misma semilla, todos obtenemos los mismos números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

---
## 1. El problema: tres almacenes, ¿tardan lo mismo?

Una empresa de comercio electrónico tiene tres almacenes (A, B y C). De cada uno tomamos 40 pedidos y medimos el **tiempo de preparación** en minutos (*ejemplo ilustrativo*, datos sintéticos). ¿Hay diferencias reales entre los almacenes o son ruido de muestreo?

Con dos grupos usarías un t-test de Welch (clase 7). Con tres, la tentación es hacer tres t-tests a pares; enseguida veremos por qué **no** es buena idea.

In [ ]:
rng_datos = np.random.default_rng(42)
tiempo_a = rng_datos.normal(32, 6, 40)      # ejemplo ilustrativo (minutos)
tiempo_b = rng_datos.normal(33, 6, 40)
tiempo_c = rng_datos.normal(38, 6, 40)
grupos = [tiempo_a, tiempo_b, tiempo_c]
nombres = ["A", "B", "C"]

resumen = pd.DataFrame({
    "n": [len(g) for g in grupos],
    "media (min)": [g.mean() for g in grupos],
    "desv. típica (min)": [g.std(ddof=1) for g in grupos]},
    index=["almacén " + nom for nom in nombres])
display(resumen.round(2))

plt.boxplot(grupos, tick_labels=nombres)
plt.title("Tiempo de preparación por almacén")
plt.ylabel("minutos")
plt.show()

---
## 2. La idea del ANOVA: varianza entre grupos frente a varianza dentro de los grupos

El ANOVA de un factor contrasta **H0: todas las medias poblacionales son iguales** frente a H1: **al menos una es distinta**. Compara dos cantidades:

- la **variabilidad entre grupos** (cuánto se alejan las medias de grupo de la media global);
- la **variabilidad dentro de los grupos** (cuánto varían los datos alrededor de la media de *su* grupo: el ruido).

El estadístico **F** es el cociente de las dos, cada una dividida por sus grados de libertad. Si H0 es cierta, F ronda 1; si las medias difieren de verdad, F crece. Primero lo calculamos **a mano** con un ejemplo minúsculo: 3 grupos de 4 pedidos.

In [ ]:
peq = {"A": np.array([12, 14, 13, 15]),
       "B": np.array([16, 18, 17, 15]),
       "C": np.array([20, 19, 22, 19])}          # ejemplo ilustrativo (minutos)

valores = np.concatenate(list(peq.values()))
media_global = valores.mean()
ss_entre = sum(len(v) * (v.mean() - media_global) ** 2 for v in peq.values())     # suma de cuadrados entre grupos
ss_dentro = sum(((v - v.mean()) ** 2).sum() for v in peq.values())                # suma de cuadrados dentro de los grupos
k, N = len(peq), len(valores)
gl_entre, gl_dentro = k - 1, N - k
ms_entre, ms_dentro = ss_entre / gl_entre, ss_dentro / gl_dentro                   # cuadrados medios
F = ms_entre / ms_dentro
p_F = stats.f.sf(F, gl_entre, gl_dentro)
eta2 = ss_entre / (ss_entre + ss_dentro)

tabla_anova = pd.DataFrame({
    "suma de cuadrados": [ss_entre, ss_dentro, ss_entre + ss_dentro],
    "gl": [gl_entre, gl_dentro, N - 1],
    "cuadrado medio": [ms_entre, ms_dentro, np.nan]},
    index=["entre grupos", "dentro de los grupos", "total"])
display(tabla_anova.round(3))
print(f"media global = {media_global:.3f}   F = {F:.3f}   p = {p_F:.5f}   η² = {eta2:.3f}")
print(f"valor crítico de F (α = 0,05) = {stats.f.ppf(0.95, gl_entre, gl_dentro):.3f}")

ref = stats.f_oneway(*peq.values())
print(f"scipy: F = {ref.statistic:.3f}   p = {ref.pvalue:.5f}")
assert np.isclose(F, ref.statistic) and np.isclose(p_F, ref.pvalue)

### Experimento guiado: el ruido dentro de los grupos decide

Mantenemos **exactamente las mismas medias de grupo** (13,5 · 16,5 · 20) pero multiplicamos por 1, 2, 4 y 8 la dispersión de los datos alrededor de cada media. ¿Qué le pasa a F y al p-value?

In [ ]:
filas = []
for factor in [1, 2, 4, 8]:
    ruidosos = [v.mean() + factor * (v - v.mean()) for v in peq.values()]
    r = stats.f_oneway(*ruidosos)
    filas.append({"factor de ruido": factor,
                  "desv. típica media dentro": np.mean([g.std(ddof=1) for g in ruidosos]),
                  "F": r.statistic, "p-value": r.pvalue})
display(pd.DataFrame(filas).round(4).set_index("factor de ruido"))

**¿Qué observas?**

- Las medias de grupo no han cambiado, pero F cae y el p-value sube a medida que crece la dispersión interna. Con factor 1, F = 23,81; con factor 8, F ≈ 0,37.
- Una diferencia entre medias solo es "grande" **en comparación con el ruido de dentro de los grupos**. Es la misma idea de la clase 7 (diferencia / error estándar), generalizada a más de dos grupos.

*(Tu respuesta aquí.)*

---
## 3. ANOVA con `scipy` y tamaño del efecto η²

Aplicamos `stats.f_oneway` a los tres almacenes. El p-value dice si las medias son compatibles con H0; **no** dice cuánto pesa el factor. Para eso usamos **η² (eta cuadrado)** = suma de cuadrados entre grupos / suma de cuadrados total: la proporción de la variabilidad total que explica la pertenencia al grupo.

In [ ]:
def eta_cuadrado(*gs):
    """η² = SS entre grupos / SS total."""
    todo = np.concatenate(gs)
    ss_total = ((todo - todo.mean()) ** 2).sum()
    ss_e = sum(len(g) * (g.mean() - todo.mean()) ** 2 for g in gs)
    return ss_e / ss_total

anova = stats.f_oneway(*grupos)
print(f"F = {anova.statistic:.3f}   p = {anova.pvalue:.6f}   η² = {eta_cuadrado(*grupos):.3f}")

**¿Qué observas?**

- El p-value es muy pequeño: rechazamos H0 (todas las medias iguales).
- η² ≈ 0,14: el almacén explica alrededor del 14 % de la variabilidad de los tiempos; el otro 86 % es variación entre pedidos dentro de cada almacén. (Como orientación habitual, 0,01 pequeño, 0,06 mediano, 0,14 grande: son convenciones, no verdades.)
- ¿Sabes ya **qué** almacenes son distintos? No: el ANOVA solo dice que **al menos uno** lo es.

*(Tu respuesta aquí.)*

---
## 4. ¿Qué grupos difieren? Comparaciones posteriores de Tukey

El **test HSD de Tukey** compara todos los pares de medias controlando la probabilidad de algún falso positivo **entre las comparaciones del conjunto** (calibrado para el rango de las medias). Da un p-value ajustado y un intervalo de confianza para cada diferencia. Se usa **después** de un ANOVA significativo, no en su lugar.

In [ ]:
tk = stats.tukey_hsd(*grupos)
ic_tk = tk.confidence_interval(confidence_level=0.95)
filas = []
for i in range(3):
    for j in range(i + 1, 3):
        filas.append({"comparación": f"{nombres[i]} − {nombres[j]}",
                      "diferencia de medias": grupos[i].mean() - grupos[j].mean(),
                      "IC 95 % inf.": ic_tk.low[i, j], "IC 95 % sup.": ic_tk.high[i, j],
                      "p ajustado (Tukey)": tk.pvalue[i, j]})
display(pd.DataFrame(filas).set_index("comparación").round(4))

# el mismo resultado con statsmodels
from statsmodels.stats.multicomp import pairwise_tukeyhsd
etiquetas = np.repeat(nombres, 40)
print(pairwise_tukeyhsd(np.concatenate(grupos), etiquetas))

**¿Qué observas?**

- A y B **no** se distinguen (p ajustado ≈ 0,70 y el IC incluye 0); C es más lento que A y que B (p ajustados ≈ 0,000 y ≈ 0,003).
- Por eso el ANOVA significativo **no** significa "todos difieren": aquí difiere uno solo, el almacén C.
- `scipy` y `statsmodels` dan los mismos p-values (el signo de las diferencias depende del orden del par).
- Si en cambio hubiéramos hecho tres t-tests sin corrección, cada uno con α = 0,05, la probabilidad de algún falso positivo con H0 cierta sería 1 − 0,95³ ≈ 14 %, no 5 %.

*(Tu respuesta aquí.)*

---
## 5. Supuestos del ANOVA y qué hacer si fallan

El ANOVA clásico supone: (1) observaciones **independientes**; (2) **residuos** (cada dato menos la media de su grupo) aproximadamente **normales**; (3) **varianzas** parecidas entre grupos. La independencia depende del diseño; las otras dos se miran con gráficos y tests:

- **Gráfico Q-Q de los residuos**: puntos cerca de la recta = normalidad razonable.
- **Shapiro-Wilk** sobre los residuos: complemento con límites (poco potente con n pequeño, hipersensible con n grande).
- **Levene**: H0 = las varianzas son iguales.

Si fallan: **Welch-ANOVA** (no exige varianzas iguales) o el test no paramétrico de **Kruskal-Wallis** (sección 6).

In [ ]:
def residuos_de(*gs):
    return np.concatenate([g - g.mean() for g in gs])

# Caso 1: los tres almacenes
res_alm = residuos_de(*grupos)
print(f"Almacenes: Shapiro-Wilk (residuos) p = {stats.shapiro(res_alm).pvalue:.3f}   Levene p = {stats.levene(*grupos).pvalue:.3f}")

# Caso 2: tiempos de espera con cola larga (ejemplo ilustrativo)
rng_cola = np.random.default_rng(3)
cola_1 = rng_cola.lognormal(3.0, 0.7, 20)
cola_2 = rng_cola.lognormal(3.1, 0.7, 20)
cola_3 = rng_cola.lognormal(3.7, 0.7, 20)
colas = [cola_1, cola_2, cola_3]
res_cola = residuos_de(*colas)
print(f"Colas largas: Shapiro-Wilk (residuos) p = {stats.shapiro(res_cola).pvalue:.1e}   Levene p = {stats.levene(*colas).pvalue:.3f}   desv. típicas = {[round(float(g.std(ddof=1)), 1) for g in colas]}")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
stats.probplot(res_alm, dist="norm", plot=ejes[0]); ejes[0].set_title("Q-Q de los residuos: almacenes")
stats.probplot(res_cola, dist="norm", plot=ejes[1]); ejes[1].set_title("Q-Q de los residuos: colas largas")
plt.tight_layout()
plt.show()

Aplicamos al segundo caso el ANOVA clásico y el **Welch-ANOVA** (`statsmodels.stats.oneway.anova_oneway` con `use_var="unequal"`), que no exige varianzas iguales:

In [ ]:
from statsmodels.stats.oneway import anova_oneway

clasico = stats.f_oneway(*colas)
welch = anova_oneway(colas, use_var="unequal", welch_correction=True)
print(f"ANOVA clásico: F = {clasico.statistic:.3f}   p = {clasico.pvalue:.4f}")
print(f"Welch-ANOVA:  F = {welch.statistic:.3f}   p = {welch.pvalue:.4f}")

**¿Qué observas?**

- En los almacenes, el Q-Q sigue la recta y ni Shapiro ni Levene rechazan: los supuestos son razonables.
- En los tiempos con cola larga, el Q-Q se curva y Shapiro rechaza con claridad. Las desviaciones típicas van de ≈ 20 a ≈ 45, pero Levene no rechaza (p ≈ 0,89): con n = 20 por grupo y datos asimétricos tiene poca potencia, así que un p alto en Levene no garantiza varianzas iguales.
- El ANOVA clásico y el Welch-ANOVA dan conclusiones distintas (p ≈ 0,10 frente a p ≈ 0,03). Cuando los supuestos fallan, la elección del test **cambia la conclusión**: hay que decidirla con criterio y, si es posible, antes de ver el p-value.

*(Tu respuesta aquí.)*

---
## 6. Tests no paramétricos: Mann-Whitney y Kruskal-Wallis

Trabajan con los **rangos** (posición de cada dato al ordenar todos juntos) y no suponen normalidad. **Mann-Whitney U** compara 2 grupos independientes; **Kruskal-Wallis** compara ≥ 3. Lo que contrastan en realidad: si un dato de un grupo tiende a ser **mayor** que uno del otro (**dominancia estocástica**, P(X > Y) ≠ 0,5), **no** "las medianas" en general. Se prefieren con colas pesadas, datos ordinales o n pequeño; si los datos son normales, pierden algo de potencia.

In [ ]:
rng_np = np.random.default_rng(7)
espera_x = rng_np.lognormal(3.0, 0.8, 25)          # ejemplo ilustrativo: tiempos de espera con cola larga
espera_y = rng_np.lognormal(3.5, 0.8, 25)
welch_x = stats.ttest_ind(espera_x, espera_y, equal_var=False)
mw = stats.mannwhitneyu(espera_x, espera_y, alternative="two-sided")
print(f"medianas: {np.median(espera_x):.1f} y {np.median(espera_y):.1f}")
print(f"t de Welch: p = {welch_x.pvalue:.4f}")
print(f"Mann-Whitney:  U = {mw.statistic:.0f}   p = {mw.pvalue:.4f}   P(X > Y) estimada = {mw.statistic / (25 * 25):.3f}")

kw = stats.kruskal(*colas)
print(f"Kruskal-Wallis (3 grupos de colas largas): H = {kw.statistic:.2f}   p = {kw.pvalue:.5f}   (ANOVA clásico p = {stats.f_oneway(*colas).pvalue:.3f})")

### Mann-Whitney no es un test sobre las medianas

Generamos dos grupos de 1 000 datos con **la misma mediana** (≈ 10) pero formas distintas: uno simétrico y otro con cola larga a la derecha.

In [ ]:
rs = np.random.default_rng(11)
sim_x = rs.normal(10, 1, 1000)
sim_y = 10 + (rs.lognormal(0, 1.0, 1000) - 1.0)
mw2 = stats.mannwhitneyu(sim_x, sim_y)
print(f"medianas: {np.median(sim_x):.2f} y {np.median(sim_y):.2f}   medias: {sim_x.mean():.2f} y {sim_y.mean():.2f}")
print(f"Mann-Whitney: p = {mw2.pvalue:.6f}   P(X > Y) = {mw2.statistic / 1000 ** 2:.3f}")
print(f"test de la mediana de Mood: p = {stats.median_test(sim_x, sim_y)[1]:.3f}")

plt.hist(sim_x, bins=40, alpha=0.6, label="X")
plt.hist(sim_y, bins=40, alpha=0.6, label="Y")
plt.title("Misma mediana, formas distintas"); plt.legend()
plt.show()

**¿Qué observas?**

- Mann-Whitney sale muy significativo (p ≈ 9 · 10⁻⁶) aunque las medianas son casi idénticas: detecta que P(X > Y) ≈ 0,44, no que las medianas difieran. El test de la mediana no encuentra nada (p ≈ 0,45).
- Interprétalo como "un dato de Y tiende a ser mayor que uno de X" (probabilidad ≈ 0,56), no como "las medianas son distintas". Solo se puede leer como diferencia de medianas si supones que las dos distribuciones tienen la **misma forma** (desplazamiento).
- En los tiempos con cola larga, Mann-Whitney y Kruskal-Wallis dan p más pequeños que sus equivalentes paramétricos: con colas pesadas suelen tener **más** potencia.

*(Tu respuesta aquí.)*

### Experimento guiado: la potencia del t de Welch frente a Mann-Whitney

Repetimos 2 000 veces una comparación con n = 20 por grupo y un efecto real, con tres formas de datos, y contamos con qué frecuencia rechaza cada test (potencia).

In [ ]:
def potencia_comparada(generador, n=20, reps=2000, semilla=1):
    r = np.random.default_rng(semilla)
    rechaza_t = rechaza_mw = 0
    for _ in range(reps):
        a, b = generador(r, n)
        rechaza_t += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
        rechaza_mw += stats.mannwhitneyu(a, b).pvalue < 0.05
    return rechaza_t / reps, rechaza_mw / reps

escenarios = {
    "Normal": lambda r, n: (r.normal(0, 1, n), r.normal(0.8, 1, n)),
    "Colas pesadas (t con 3 gl)": lambda r, n: (r.standard_t(3, n), r.standard_t(3, n) + 1.2),
    "Asimétrica (lognormal)": lambda r, n: (r.lognormal(0, 1, n), r.lognormal(0.8, 1, n)),
}
filas = []
for nombre, gen in escenarios.items():
    pt, pmw = potencia_comparada(gen)
    filas.append({"escenario": nombre, "potencia t de Welch": pt, "potencia Mann-Whitney": pmw})
display(pd.DataFrame(filas).set_index("escenario").round(3))

**¿Qué observas?**

- Con datos normales, las dos potencias son casi iguales (≈ 0,70 y ≈ 0,69): Mann-Whitney pierde muy poco.
- Con colas pesadas o asimetría, Mann-Whitney detecta el efecto bastante más a menudo (≈ 0,81 frente a ≈ 0,66; ≈ 0,69 frente a ≈ 0,48).
- "No paramétrico" no significa "peor": cuando los supuestos del t-test son dudosos, suele ser mejor.

*(Tu respuesta aquí.)*

---
## 7. ¿ANOVA o Kruskal-Wallis? Una simulación con datos de cola larga

Con **tres grupos** de 15 datos comparamos la frecuencia de rechazo de los dos tests en tres formas de datos. Para cada forma simulamos 2 000 "estudios" en dos situaciones: **sin efecto** (H0 cierta: la frecuencia de rechazo es la tasa de falsos positivos, que debería rondar 0,05) y **con un efecto real** (la frecuencia de rechazo es la potencia).

In [ ]:
def comparar_anova_kw(generador, n=15, reps=2000, semilla=5):
    """Frecuencia de rechazo (p < 0,05) de ANOVA y de Kruskal-Wallis."""
    r = np.random.default_rng(semilla)
    rechaza_f = rechaza_kw = 0
    for _ in range(reps):
        gs = generador(r, n)
        rechaza_f += stats.f_oneway(*gs).pvalue < 0.05
        rechaza_kw += stats.kruskal(*gs).pvalue < 0.05
    return rechaza_f / reps, rechaza_kw / reps

# (sin efecto, con efecto) para cada forma de datos
escenarios3 = {
    "Normal": (lambda r, n: [r.normal(0, 1, n) for _ in range(3)],
                         lambda r, n: [r.normal(m, 1, n) for m in (0, 0.5, 1.0)]),
    "Colas pesadas (t con 3 gl)": (lambda r, n: [r.standard_t(3, n) for _ in range(3)],
                                                             lambda r, n: [r.standard_t(3, n) + m for m in (0, 0.8, 1.6)]),
    "Asimétrica (lognormal)": (lambda r, n: [r.lognormal(0, 1, n) for _ in range(3)],
                                                       lambda r, n: [r.lognormal(m, 1, n) for m in (0, 0.5, 1.0)]),
}
filas = []
for nombre, (sin_efecto, con_efecto) in escenarios3.items():
    fp_f, fp_kw = comparar_anova_kw(sin_efecto)
    pot_f, pot_kw = comparar_anova_kw(con_efecto)
    filas.append({"datos": nombre,
                  "falsos positivos ANOVA": fp_f, "falsos positivos Kruskal-Wallis": fp_kw,
                  "potencia ANOVA": pot_f, "potencia Kruskal-Wallis": pot_kw})
tabla_pot = pd.DataFrame(filas).set_index("datos")
display(tabla_pot.round(3))

tabla_pot.iloc[:, 2:].plot.bar(rot=0, figsize=(9, 4))
plt.title("Potencia: ANOVA frente a Kruskal-Wallis (3 grupos, n = 15)")
plt.ylabel("potencia")
plt.show()

**¿Qué observas?**

- Sin efecto, los dos tests rechazan entre el 3 % y el 5 % de las veces: ambos controlan el error de tipo I también con datos asimétricos (con n = 15 la aproximación del ANOVA es razonable).
- Con datos normales las potencias son parecidas (≈ 0,65 y ≈ 0,61): Kruskal-Wallis pierde poco.
- Con colas pesadas o asimetría, Kruskal-Wallis **gana** (≈ 0,81 frente a ≈ 0,68 con t de 3 gl; ≈ 0,61 frente a ≈ 0,42 con lognormal). Con un ANOVA se perderían efectos reales.
- Ojo: esto vale para estas formas y estos efectos. Elige el test **antes** de mirar los p-values; si pruebas los dos y te quedas con el más pequeño estás haciendo dos contrastes y quedándote con el mejor.

*(Tu respuesta aquí.)*

---
## 8. Leer y comunicar un resultado completo

Un buen informe de la comparación de grupos reúne **cuatro elementos**: el resultado global (F con sus grados de libertad y p-value), el tamaño del efecto (η²), las medias (y desviaciones) de cada grupo y el post-hoc con la diferencia, su intervalo de confianza y el p ajustado. Generamos el texto a partir de los resultados de los almacenes (sección 3-4) y del Kruskal-Wallis de las colas largas (sección 5).

In [ ]:
k = len(grupos)
N = sum(len(g) for g in grupos)
anova = stats.f_oneway(*grupos)
lineas = [f"ANOVA: F({k - 1}, {N - k}) = {anova.statistic:.2f}, p = {anova.pvalue:.4f}, η² = {eta_cuadrado(*grupos):.2f}.",
          "Medias (min): " + ", ".join(f"{nombres[i]} = {grupos[i].mean():.1f} ± {grupos[i].std(ddof=1):.1f}" for i in range(k)) + "."]
for i in range(k):
    for j in range(i + 1, k):
        dif = grupos[i].mean() - grupos[j].mean()
        veredicto = "difieren" if tk.pvalue[i, j] < 0.05 else "no se distinguen"
        lineas.append(f"Tukey {nombres[i]} − {nombres[j]}: {dif:+.2f} min, IC 95 % ({ic_tk.low[i, j]:.2f}; {ic_tk.high[i, j]:.2f}), p ajustado = {tk.pvalue[i, j]:.4f} → {veredicto}.")
print("\n".join(lineas))

# Versión no paramétrica con las colas largas
kw = stats.kruskal(*colas)
print(f"\nKruskal-Wallis: H({len(colas) - 1}) = {kw.statistic:.2f}, p = {kw.pvalue:.4f}. Medianas: " + ", ".join(f"{np.median(g):.1f}" for g in colas) + ".")

**¿Qué observas?**

- Del resultado global (F, p) pasamos a los pares: solo los pares con C difieren; A − B tiene un IC que incluye 0.
- El párrafo de conclusión sería: *"El tiempo de preparación difiere entre almacenes (ANOVA, F(2, 117) = 9,78; p = 0,0001; η² = 0,14, efecto grande). Tukey indica que C es más lento que A (≈ 4,4 min) y que B (≈ 3,5 min); A y B no se distinguen."* Se dice **qué** difiere, **cuánto** y con qué incertidumbre.
- Un informe que solo dice "p < 0,05, hay diferencias significativas" no permite decidir nada.

*(Tu respuesta aquí.)*

---
## 9. Cómo elegir el test para comparar grupos numéricos

| Situación | Supuestos razonables | Supuestos dudosos (colas pesadas, valores extremos, ordinales, n pequeño) |
|---|---|---|
| 2 grupos independientes | t de Welch (clase 7) | Mann-Whitney (`stats.mannwhitneyu`) |
| 2 medidas del mismo sujeto (pareadas) | t pareado (clase 7) | Wilcoxon de rangos con signo (`stats.wilcoxon`) |
| ≥ 3 grupos independientes | ANOVA + Tukey; Welch-ANOVA si las varianzas difieren | Kruskal-Wallis (+ Dunn como post-hoc) |
| Variables categóricas (conteos) | clase 9: chi-cuadrado, Fisher | — |
| Muchos tests a la vez | clase 9: Bonferroni, Benjamini-Hochberg | — |

Tukey ya corrige por el número de pares comparados: **no** hace falta añadir Bonferroni encima. Si haces varios Mann-Whitney a pares (porque no hay Dunn en `scipy`), sí necesitas una corrección de la clase 9.

---
## Resumen de la clase

- **Muchos t-tests a pares** inflan el riesgo de falsos positivos (1 − 0,95³ ≈ 14 % con 3 grupos). Un único test global lo evita.
- **ANOVA de un factor** (`f_oneway`): F = varianza entre grupos / varianza dentro; H0: todas las medias iguales. Informa también **η²**. Un ANOVA significativo dice que **al menos un** grupo difiere, no cuáles: usa **Tukey** (`stats.tukey_hsd`), que ya controla el error del conjunto de pares.
- **Supuestos**: independencia, residuos aproximadamente normales (Q-Q; Shapiro con sus límites) y varianzas parecidas (Levene, que puede tener poca potencia). Si fallan: Welch-ANOVA o Kruskal-Wallis.
- **Mann-Whitney y Kruskal-Wallis** comparan rangos (dominancia estocástica), no "las medianas" sin condiciones; ganan con colas pesadas y pierden poco con datos normales.
- Un buen informe reúne **F(gl), p, η², medias y post-hoc con IC**; el test se elige **antes** de ver los p-values.

**Siguiente paso:** el notebook de práctica de la clase 8 y, después, la clase 9 (categorías y comparaciones múltiples).